In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("WebLogAnalysis").getOrCreate()
# Đọc file web_logs.csv
df_logs = spark.read.csv("data/web_logs.csv", header=True, inferSchema=True)
# Hiển thị schema
print("\n. Cấu trúc Schema:")
df_logs.printSchema()
# Đếm tổng số bản ghi
total_records = df_logs.count()
print(f"\n. Tổng số bản ghi: {total_records}")


. Cấu trúc Schema:
root
 |-- timestamp: timestamp (nullable = true)
 |-- user_id: string (nullable = true)
 |-- page: string (nullable = true)
 |-- action: string (nullable = true)
 |-- device: string (nullable = true)
 |-- country: string (nullable = true)


. Tổng số bản ghi: 4


In [ ]:
# Phần B Phân tích cơ bản

In [5]:
from pyspark.sql.functions import desc
# Đếm số lượt truy cập theo page
df_logs.groupBy("page").count().show()
# Tìm 5 trang được truy cập nhiều nhất
df_logs.groupBy("page").count().orderBy(desc("count")).show(5)
# Đếm số lượt truy cập theo device
df_logs.groupBy("device").count().show()
# Đếm số lượt truy cập theo country
df_logs.groupBy("country").count().show()

+-------+-----+
|   page|count|
+-------+-----+
|   cart|    1|
|   home|    2|
|product|    1|
+-------+-----+

+-------+-----+
|   page|count|
+-------+-----+
|   home|    2|
|   cart|    1|
|product|    1|
+-------+-----+

+-------+-----+
| device|count|
+-------+-----+
|desktop|    1|
| mobile|    2|
| tablet|    1|
+-------+-----+

+--------+-----+
| country|count|
+--------+-----+
|Thailand|    1|
| Vietnam|    3|
+--------+-----+



In [ ]:
# Phần C Phân tích người dùng

In [7]:
from pyspark.sql.functions import countDistinct
# Tính số lượng người dùng duy nhất
df_logs.select(countDistinct("user_id")).show()
# top 10 người truy cập nhiều nhất
df_logs.groupBy("user_id").count().orderBy(desc("count")).show(10)

+-----------------------+
|count(DISTINCT user_id)|
+-----------------------+
|                      3|
+-----------------------+

+-------+-----+
|user_id|count|
+-------+-----+
|   U001|    2|
|   U002|    1|
|   U003|    1|
+-------+-----+



In [ ]:
# Phần D Phân tích nâng cao

In [17]:
from pyspark.sql.functions import to_date, col
# Tạo cột Date từ timestamp
df_logs =df_logs.withColumn("date", to_date(col("timestamp")))
df_logs.show()
# Tính số lượt truy cập theo ngày
df_logs.groupBy("date").count().orderBy("date").show()

+-------------------+-------+-------+------+-------+--------+----------+
|          timestamp|user_id|   page|action| device| country|      date|
+-------------------+-------+-------+------+-------+--------+----------+
|2026-04-01 08:00:00|   U001|   home|  view| mobile| Vietnam|2026-04-01|
|2026-04-01 08:01:10|   U002|product| click|desktop| Vietnam|2026-04-01|
|2026-04-01 08:02:15|   U001|   cart|  view| mobile| Vietnam|2026-04-01|
|2026-04-01 08:03:00|   U003|   home|  view| tablet|Thailand|2026-04-01|
+-------------------+-------+-------+------+-------+--------+----------+

+----------+-----+
|      date|count|
+----------+-----+
|2026-04-01|    4|
+----------+-----+



Câu 1: Log web thể hiện đặc điểm nào của Big Data: volume, velocity, variety?
Log web thực tế thể hiện cực kỳ rõ nét cả 3 đặc điểm (3V) cốt lõi này của Big Data:

Volume (Dung lượng lớn): Hệ thống ghi lại mọi thao tác nhỏ nhất (click, view, cuộn trang...). Với các trang web thương mại điện tử lớn, dung lượng file log sinh ra mỗi ngày có thể lên tới hàng chục, hàng trăm Terabyte.

Velocity (Tốc độ cao): Dữ liệu log không đứng im mà được sinh ra liên tục, chớp nhoáng theo từng mili-giây song song với lượng truy cập của hàng triệu người dùng cùng lúc.

Variety (Đa dạng): Mặc dù trong bài tập này dữ liệu đã được làm sạch thành dạng bảng (CSV), nhưng log thô nguyên bản trong thực tế thường rất lộn xộn, nằm ở dạng bán cấu trúc (như JSON, XML) hoặc phi cấu trúc (dòng văn bản raw text).

Câu 2: Trong phân tích hành vi người dùng, việc đếm theo page và device mang ý nghĩa gì?

Đếm theo page (Trang): Giúp doanh nghiệp xác định đâu là nội dung/sản phẩm đang "hot" nhất và thu hút nhiều sự chú ý nhất. Từ đó, họ có thể đưa ra quyết định kinh doanh như: đưa sản phẩm đó lên trang chủ, chạy quảng cáo thêm, hoặc phân tích xem tại sao các trang khác lại ít người xem.

Đếm theo device (Thiết bị): Vẽ ra bức tranh về thói quen của người dùng (họ thích lướt web bằng Điện thoại, Máy tính hay Tablet). Ví dụ: Nếu thấy 80% lượt truy cập là từ mobile, đội ngũ lập trình bắt buộc phải ưu tiên thiết kế giao diện "Mobile-first" để website chạy mượt mà nhất trên màn hình nhỏ.

Câu 3: Nếu log được sinh ra liên tục theo thời gian thực, bài toán này sẽ chuyển sang hướng nào?
Bài toán sẽ lập tức chuyển từ Xử lý theo lô (Batch Processing) sang Xử lý dữ liệu luồng (Stream Processing / Real-time Analytics).

Thay vì đợi dữ liệu gom lại thành một file CSV tĩnh rồi mới đọc một lần, hệ thống sẽ phải "hứng" dữ liệu trực tiếp khi nó vừa sinh ra (thường kết hợp với các hệ thống như Apache Kafka).

Trong lập trình PySpark, chúng ta sẽ phải từ bỏ các lệnh đọc tĩnh (spark.read) và chuyển sang sử dụng công nghệ Spark Structured Streaming (bằng các lệnh như spark.readStream). Khi đó, các bảng thống kê lượt truy cập hay doanh thu sẽ nhảy số và cập nhật liên tục từng giây ngay trên màn hình (dashboard) chứ không đợi chạy lại code.